# Lab 08 — Class Imbalance & Cost-Sensitive Learning

*AI Cybersecurity Masterclass · Lab 08 · AI60secLab*

**Companion video:** [▶ One Number Decides: Attack or Not?](https://www.youtube.com/watch?v=7yn55E7UxZ4&list=PLG4l6Ssi9A00)

**Research question:** What happens when attacks become rare in the training data?

We keep the official test set untouched and compare:
- no class weighting;
- balanced class weighting;
- controlled random undersampling.

## Security foundation: the base-rate problem

Real attacks are **rare**. Suppose 1 flow in 10,000 is malicious and your detector catches 99% of attacks with a 1% false-alarm rate. Out of 1,000,000 flows:
- about 100 are attacks, and it catches 99 of them;
- about 999,900 are normal, and 1% of those, about 9,999, raise false alarms.

So only about **1 alert in 100 is real**, even with a detector that sounds excellent. This is the **base-rate fallacy**, and it is the main reason SOCs drown in alerts.

UNSW-NB15 hides this problem because most of its rows are attacks. The techniques in this lab (class weights, cost-sensitive learning) matter far more on real traffic.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [2]:
import pandas as pd

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame
from src.imbalance import (
    class_balance_frame,
    make_controlled_binary_imbalance,
    random_undersample_majority,
    balanced_class_weight_dict,
)

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [3]:
X_imbalanced, y_imbalanced = make_controlled_binary_imbalance(
    X_train,
    y_train,
    positive_to_negative_ratio=0.10,
    random_state=42,
)

print("Original:")
display(class_balance_frame(y_train))

print("Controlled rare-attack training set:")
display(class_balance_frame(y_imbalanced))

print("Balanced class weights:")
print(balanced_class_weight_dict(y_imbalanced))

Original:


,count,fraction
label,,
0,56000,0.319378
1,119341,0.680622


Controlled rare-attack training set:


,count,fraction
label,,
0,56000,0.909091
1,5600,0.090909


Balanced class weights:
{np.int64(0): np.float64(0.55), np.int64(1): np.float64(5.5)}


In [4]:
unweighted = logistic_pipeline(
    X_imbalanced,
    class_weight=None,
)
unweighted.fit(X_imbalanced, y_imbalanced)

pred_unweighted = unweighted.predict(X_test)
score_unweighted = unweighted.predict_proba(X_test)[:, 1]
m_unweighted = binary_metrics(
    y_test,
    pred_unweighted,
    score_unweighted,
)

In [5]:
weighted = logistic_pipeline(
    X_imbalanced,
    class_weight="balanced",
)
weighted.fit(X_imbalanced, y_imbalanced)

pred_weighted = weighted.predict(X_test)
score_weighted = weighted.predict_proba(X_test)[:, 1]
m_weighted = binary_metrics(
    y_test,
    pred_weighted,
    score_weighted,
)

In [6]:
X_under, y_under = random_undersample_majority(
    X_imbalanced,
    y_imbalanced,
    target_ratio=0.50,
    random_state=42,
)

undersampled = logistic_pipeline(
    X_under,
    class_weight=None,
)
undersampled.fit(X_under, y_under)

pred_under = undersampled.predict(X_test)
score_under = undersampled.predict_proba(X_test)[:, 1]
m_under = binary_metrics(
    y_test,
    pred_under,
    score_under,
)

In [7]:
comparison = pd.concat([
    binary_metrics_frame("Unweighted", m_unweighted),
    binary_metrics_frame("Class weighted", m_weighted),
    binary_metrics_frame("Undersampled", m_under),
])

display(
    comparison[
        [
            "precision",
            "recall",
            "f1",
            "false_positive_rate",
            "false_negative_rate",
            "roc_auc",
        ]
    ].round(4)
)

,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,
Unweighted,0.9951,0.7157,0.8326,0.0043,0.2843,0.9544
Class weighted,0.8063,0.9288,0.8632,0.2734,0.0712,0.9554
Undersampled,0.8997,0.8612,0.8800,0.1176,0.1388,0.9545


## Explain the result

- Which method recovered attack recall?
- What happened to precision?
- Which method had the lowest false-negative rate?
- What information did undersampling discard?
- Why must balancing occur only inside training data?

## From lab to SOC

When evaluating any detector, ask for the **alerts per day** and the **share of alerts that are real** on traffic like yours, not just recall. That is the number your analysts will live with.

**Check yourself**
1. Redo the base-rate example with a 0.1% false-alarm rate. What share of alerts is real now?
2. Why do class weights help a model care about rare attacks?
3. Why does precision measured on UNSW-NB15 overstate production precision?